In [1]:
import numpy as np
import pandas as pd
import torch

from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().resolve().parents[0] / '2_Propensities'))

import MF_class as MF

np.random.seed(42)
if np.random.choice(np.arange(1000)) != 102:
    raise ValueError("Random seed is not set correctly.")

```
                                USERS                             
         ┌───────────────────────────────────────────────────────┐
         │                          │                            │
         │                          │                            │
         │                          │                            │
         │                          │                            │
ITEMS    │                          │                            │
         │                          │                            │
         │                          │                            │
         ├──────────────────────────┼────────────────────────────┤
         │                          │████████████████████████████│
         │                          │████████████████████████████│
         └───────────────────────────────────────────────────────┘
```

# 1. Load Data

In [2]:
base_artifacts = Path.cwd().resolve().parents[1] / 'CausalI2I_artifacts'
data_path = base_artifacts / 'Datasets' / 'Sequels'

train = pd.read_csv(data_path / 'train.csv')
test = pd.read_csv(data_path / 'test.csv')

n_users = train['user_id'].nunique()
n_items = train['item_id'].nunique()
print(f'Number of users: {n_users}, Number of items: {n_items}')

Number of users: 7801, Number of items: 6384


# 2. Train Model

In [3]:
model = MF.MatrixFactorizationTorch(n_users, n_items, n_factors=25)
model.fit(
    train_data=train.values,
    val_data=test.values,
    lr=5e-4, 
    wd=1e-7,
    pos_weight=1,
    batch_size=2**15,
    n_epochs=40,
    device=torch.device('cuda:0'), 
    use_amp=True)

Epoch  ||- - - - - - - - Train - - - - - - - -||- - - - - - Validation - - - - - - - || Epoch's | COS θ | Time     
Number || BCE    | BCE-POS | BCE-NEG | MPR    || BCE    | BCE-POS | BCE-NEG | MPR    || Change  |       | Elapsed  
=======||========|=========|=========|========||========|=========|=========|========||=========|=======|==========
   1   || 0.0784 |  3.1368 |  0.0294 | 0.7646 || 0.1342 |  2.6344 |  0.0494 | 0.7701 || 193.87  | None  | 00:09.07
   2   || 0.0713 |  3.4858 |  0.0167 | 0.7824 || 0.1259 |  2.8652 |  0.0331 | 0.7787 ||  34.09  | 0.834 | 00:17.60
   3   || 0.0709 |  3.4906 |  0.0161 | 0.7850 || 0.1254 |  2.8680 |  0.0325 | 0.7796 ||   9.87  | 0.734 | 00:26.20
   4   || 0.0706 |  3.4806 |  0.0160 | 0.7872 || 0.1251 |  2.8628 |  0.0323 | 0.7807 ||   8.82  | 0.628 | 00:35.20
   5   || 0.0702 |  3.4570 |  0.0159 | 0.7913 || 0.1245 |  2.8415 |  0.0324 | 0.7838 ||  12.59  | 0.822 | 00:44.06
   6   || 0.0692 |  3.4063 |  0.0158 | 0.8001 || 0.1231 |  2.8028 |  0.0323 |

### Save Model

In [4]:
model.save(path=base_artifacts / 'Propensity_Models' / 'MF_sequels.pt', note=None)

### Load Model

In [5]:
loaded_model = MF.MatrixFactorizationTorch(n_users, n_items, n_factors=25)
loaded_model.load(path=base_artifacts / 'Propensity_Models' / 'MF_sequels.pt')

Loaded model summary:
Model:                      MatrixFactorizationTorch
Number of users:            7801
Number of items:            6384
Number of factors:          25
Learning rate:              0.0005
Weight decay:               1e-07
Positive weight:            1
Batch size:                 32768
Number of epochs:           40
Device:                     cuda:0
Use AMP:                    True
Timestamp:                  2026-04-11 14:17:00
